# Phase 3: Per-Class Metrics Analysis

## Objective
Re-evaluate all 30 saved models to compute **per-class metrics** (F1, Precision, Recall) for both validation and test sets, then generate comprehensive statistical reports.

## What This Notebook Does
1. Loads all 30 result JSON files from Phase 3
2. For each seed:
   - Loads the saved model checkpoint
   - Recreates data splits (using same seed)
   - Re-evaluates on validation (at best epoch) and test sets
   - Computes per-class F1, Precision, Recall
   - Updates JSON files with new metrics
3. Computes statistics across all 30 runs:
   - Overall metrics: Mean ± Std, CI, Min/Max, CV
   - Per-class metrics: Mean ± Std for each of 14 classes
4. Generates comprehensive reports and visualizations

## Prerequisites
- All 30 Phase 3 experiments must be completed
- Model checkpoints must exist in `experiments/phase3_robustness/artifacts/models/`
- Result JSON files must exist in `experiments/phase3_robustness/metrics/experiments/`


## 1. Setup and Configuration


In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from PIL import Image
import pandas as pd
import numpy as np
import json
import os

# Repository root (chdir so relative paths work from notebooks/)
_walk = os.path.abspath(os.getcwd())
for _ in range(10):
    if os.path.isdir(os.path.join(_walk, 'experiments')) and os.path.isdir(os.path.join(_walk, 'data')):
        PROJECT_ROOT = _walk
        break
    _walk = os.path.dirname(_walk)
else:
    PROJECT_ROOT = os.path.abspath(os.getcwd())
os.chdir(PROJECT_ROOT)
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
import warnings
from urllib.parse import unquote
from scipy import stats
from datetime import datetime

# Suppress warnings
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings('ignore')

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# ============================================
# CONFIGURATION (must match Phase 3)
# ============================================
LEARNING_RATE = 5e-5
BATCH_SIZE = 32
EARLY_STOPPING_PATIENCE = 5
DROPOUT = 0.5
WEIGHT_DECAY = 1e-4
MAX_EPOCHS = 20
MODEL_INIT_SEED = 42  # Fixed seed for model initialization

# Data split ratios (must match Phase 3)
TRAIN_RATIO = 0.7
VAL_RATIO = 0.15
TEST_RATIO = 0.15

# Experiment layout (must match Phase 3)
EXPERIMENT_ROOT = "experiments/phase3_robustness"
METRICS_DIR = os.path.join(EXPERIMENT_ROOT, "metrics")
ARTIFACTS_DIR = os.path.join(EXPERIMENT_ROOT, "artifacts")

print(f"✅ Configuration loaded")
print(f"   Experiment: {EXPERIMENT_ROOT}")
print(f"   metrics: {METRICS_DIR}")
print(f"   artifacts: {ARTIFACTS_DIR}")


/home/ding-zhang/anaconda3/envs/tf_gpu/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda
✅ Configuration loaded
   Results directory: phase3_robustness_results
   Base directory: /home/ding-zhang/Dongmei/DATA255/Project


## 2. Load Seeds List


In [2]:
# Load seeds list from Phase 3
seeds_file = os.path.join(METRICS_DIR, "seeds_list.txt")
if os.path.exists(seeds_file):
    with open(seeds_file, 'r') as f:
        lines = f.readlines()
    # Extract seed values from file
    SEEDS = []
    for line in lines:
        if 'Seed' in line and line.strip().startswith(tuple('0123456789')):
            seed_value = int(line.split('Seed')[1].strip())
            SEEDS.append(seed_value)
    SEEDS = sorted(SEEDS)
    print(f"✅ Loaded {len(SEEDS)} seeds from {seeds_file}")
    print(f"   Seeds: {SEEDS}")
else:
    print(f"⚠️  Seeds file not found: {seeds_file}")
    print("   Please ensure Phase 3 experiments have been run.")
    SEEDS = []


✅ Loaded 30 seeds from phase3_robustness_results/seeds_list.txt
   Seeds: [13, 14, 16, 17, 45, 48, 53, 58, 72, 102, 112, 115, 120, 126, 141, 215, 217, 259, 280, 288, 303, 309, 328, 333, 347, 360, 367, 378, 380, 457]


## 3. Load Full Dataset


In [3]:
# Load caption dataset
print("Loading caption dataset...")
df = pd.read_csv(os.path.join(PROJECT_ROOT, 'data', 'processed', 'caption_dataset_final_full.csv'))

# Filter for successful captions
df_success = df[df['status'] == 'success'].copy()
print(f"Total images with captions: {len(df_success)}")

# Extract style categories
df_success['style'] = df_success['style'].str.strip()  # Clean whitespace
all_styles = sorted(df_success['style'].unique())
style_to_idx = {style: idx for idx, style in enumerate(all_styles)}
idx_to_style = {idx: style for style, idx in style_to_idx.items()}
num_classes = len(all_styles)

print(f"\nStyle categories: {num_classes}")
print(f"Styles: {all_styles}")

# Use full dataset
df_full = df_success.copy()
print(f"\nFull dataset size: {len(df_full)}")

# Create caption dictionary
captions_dict = {}
for _, row in df_full.iterrows():
    captions_dict[row['image_path']] = row['caption']

print(f"\nCaption dictionary created: {len(captions_dict)} entries")


Loading caption dataset...
Total images with captions: 13230

Style categories: 14
Styles: ['conservative', 'dressy', 'ethnic', 'fairy', 'feminine', 'gal', 'girlish', 'kireime-casual', 'lolita', 'mode', 'natural', 'retro', 'rock', 'street']

Full dataset size: 13230

Caption dictionary created: 13230 entries


## 4. Load Pre-trained Models


In [4]:
# Load CLIP model
import clip
print("Loading CLIP model...")
clip_model, clip_preprocess = clip.load("ViT-B/32", device=device)
clip_model.eval()
print("CLIP model loaded!")

# Load FashionBERT
from transformers import AutoTokenizer, AutoModel
print("Loading FashionBERT...")
fashionbert_tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
fashionbert_model = AutoModel.from_pretrained('bert-base-uncased').to(device)
fashionbert_model.eval()
print("FashionBERT model loaded!")


Loading CLIP model...
CLIP model loaded!
Loading FashionBERT...


2025-11-18 03:11:23.887331: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-18 03:11:23.908290: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


FashionBERT model loaded!


2025-11-18 03:11:24.532821: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


## 5. Dataset and Model Classes (Same as Phase 3)


In [5]:
class FashionMultiModalDataset(Dataset):
    """Dataset class for multi-modal fashion style classification (same as Phase 3)"""
    
    def __init__(self, df, captions_dict, style_to_idx, transform=None, base_dir=None):
        self.df = df.reset_index(drop=True)
        self.captions_dict = captions_dict
        self.style_to_idx = style_to_idx
        self.transform = transform
        self.base_dir = base_dir
        
        # Filter out images without captions and store valid indices
        self.valid_indices = []
        for idx in range(len(self.df)):
            row = self.df.iloc[idx]
            image_path = row['image_path']
            # Convert to absolute path if needed
            if base_dir and not os.path.isabs(image_path):
                image_path = os.path.join(base_dir, image_path)
            # Decode URL-encoded characters
            if '%' in image_path:
                path_parts = image_path.split('/')
                decoded_parts = [unquote(part) if '%' in part else part for part in path_parts]
                image_path = '/'.join(decoded_parts)
            
            if image_path in captions_dict or row['image_path'] in captions_dict:
                self.valid_indices.append(idx)
        
        print(f"Dataset initialized with {len(self.valid_indices)} valid samples (out of {len(self.df)})")
    
    def __len__(self):
        return len(self.valid_indices)
    
    def __getitem__(self, idx):
        actual_idx = self.valid_indices[idx]
        row = self.df.iloc[actual_idx]
        
        # Load image
        image_path = row['image_path']
        
        # Convert to absolute path if needed
        if self.base_dir and not os.path.isabs(image_path):
            image_path = os.path.join(self.base_dir, image_path)
        
        # Decode URL-encoded characters in filename
        if '%' in image_path:
            path_parts = image_path.split('/')
            decoded_parts = [unquote(part) if '%' in part else part for part in path_parts]
            image_path = '/'.join(decoded_parts)
        
        try:
            if os.path.exists(image_path):
                image = Image.open(image_path).convert('RGB')
                if self.transform:
                    image = self.transform(image)
            else:
                image = torch.zeros(3, 224, 224)
        except Exception as e:
            image = torch.zeros(3, 224, 224)
        
        # Get caption
        caption = self.captions_dict.get(image_path, 
                                        self.captions_dict.get(row['image_path'], 
                                                              "No caption available"))
        
        # Get label
        style = row['style']
        label = self.style_to_idx[style]
        
        return {
            'image': image,
            'caption': caption,
            'label': label,
            'style': style,
            'image_path': image_path
        }

class AttentionFusion(nn.Module):
    """Self-attention fusion mechanism (same as Phase 3)"""
    
    def __init__(self, visual_dim, textual_dim, hidden_dim=512):
        super(AttentionFusion, self).__init__()
        self.visual_dim = visual_dim
        self.textual_dim = textual_dim
        self.hidden_dim = hidden_dim
        
        # Project visual features
        self.visual_proj = nn.Linear(visual_dim, hidden_dim)
        
        # Project textual features
        self.textual_proj = nn.Linear(textual_dim, hidden_dim)
        
        # Self-attention mechanism
        self.attention = nn.MultiheadAttention(hidden_dim, num_heads=8, batch_first=True)
        
        # Layer normalization
        self.layer_norm = nn.LayerNorm(hidden_dim)
        
        # Final projection
        self.final_proj = nn.Linear(hidden_dim, hidden_dim)
        
    def forward(self, visual_features, textual_features):
        # Project features to common dimension
        visual_proj = self.visual_proj(visual_features)
        textual_proj = self.textual_proj(textual_features)
        
        # Stack features for attention
        combined_features = torch.stack([visual_proj, textual_proj], dim=1)  # [batch, 2, hidden_dim]
        
        # Apply self-attention
        attended_features, _ = self.attention(combined_features, combined_features, combined_features)
        
        # Layer normalization
        attended_features = self.layer_norm(attended_features)
        
        # Average pooling across modalities
        fused_features = torch.mean(attended_features, dim=1)  # [batch, hidden_dim]
        
        # Final projection
        fused_features = self.final_proj(fused_features)
        
        return fused_features

class MultiModalFashionClassifier(nn.Module):
    """Multi-modal fashion style classifier (same as Phase 3)"""
    
    def __init__(self, clip_model, fashionbert_model, num_classes, dropout=0.5, 
                 visual_dim=512, textual_dim=768):
        super(MultiModalFashionClassifier, self).__init__()
        
        self.clip_model = clip_model
        self.fashionbert_model = fashionbert_model
        self.visual_dim = visual_dim
        self.textual_dim = textual_dim
        
        # Fusion mechanism (self-attention)
        self.fusion = AttentionFusion(visual_dim, textual_dim)
        
        # Classification head with configurable dropout
        self.classifier = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )
        
    def forward(self, images, captions):
        # Extract visual features using CLIP
        with torch.no_grad():
            visual_features = self.clip_model.encode_image(images)
            visual_features = visual_features.float()
        
        # Extract textual features using FashionBERT
        with torch.no_grad():
            inputs = fashionbert_tokenizer(
                captions, 
                return_tensors='pt', 
                padding=True, 
                truncation=True, 
                max_length=128
            ).to(device)
            outputs = self.fashionbert_model(**inputs)
            textual_features = outputs.last_hidden_state[:, 0, :]  # [CLS] token
        
        # Fuse features
        fused_features = self.fusion(visual_features, textual_features)
        
        # Classification
        logits = self.classifier(fused_features)
        
        return logits

print("✅ Model classes defined!")


✅ Model classes defined!


## 6. Evaluation Functions


In [6]:
def evaluate_model(model, data_loader, criterion, device, all_styles):
    """
    Evaluate model and return all metrics including per-class metrics
    
    Returns:
        loss, accuracy, predictions, labels, macro_f1, macro_precision, macro_recall,
        per_class_f1, per_class_precision, per_class_recall
    """
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    all_predictions = []
    all_labels = []
    
    with torch.no_grad():
        for batch in tqdm(data_loader, desc="Evaluating", leave=False):
            images = batch['image'].to(device)
            captions = batch['caption']
            labels = batch['label'].to(device)
            
            logits = model(images, captions)
            loss = criterion(logits, labels)
            
            total_loss += loss.item()
            _, predicted = torch.max(logits.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    # Calculate overall metrics
    accuracy = 100.0 * correct / total if total > 0 else 0.0
    avg_loss = total_loss / len(data_loader) if len(data_loader) > 0 else 0.0
    
    # Macro-averaged metrics
    macro_f1 = f1_score(all_labels, all_predictions, average='macro', zero_division=0)
    macro_precision = precision_score(all_labels, all_predictions, average='macro', zero_division=0)
    macro_recall = recall_score(all_labels, all_predictions, average='macro', zero_division=0)
    
    # Per-class metrics
    per_class_f1 = f1_score(all_labels, all_predictions, average=None, zero_division=0)
    per_class_precision = precision_score(all_labels, all_predictions, average=None, zero_division=0)
    per_class_recall = recall_score(all_labels, all_predictions, average=None, zero_division=0)
    
    # Convert to dictionaries with style names
    per_class_f1_dict = {all_styles[i]: float(per_class_f1[i]) for i in range(len(all_styles))}
    per_class_precision_dict = {all_styles[i]: float(per_class_precision[i]) for i in range(len(all_styles))}
    per_class_recall_dict = {all_styles[i]: float(per_class_recall[i]) for i in range(len(all_styles))}
    
    return (
        avg_loss, accuracy, all_predictions, all_labels,
        macro_f1, macro_precision, macro_recall,
        per_class_f1_dict, per_class_precision_dict, per_class_recall_dict
    )

print("✅ Evaluation function defined!")


✅ Evaluation function defined!


## 7. Main Function: Evaluate Seed and Add Per-Class Metrics


In [7]:
def evaluate_seed_with_per_class_metrics(seed_value, seed_idx, df_full, captions_dict, style_to_idx,
                                          clip_model, fashionbert_model, fashionbert_tokenizer,
                                          num_classes, device, all_styles, base_dir):
    """
    Evaluate a single seed's model and add per-class metrics to results
    
    Args:
        seed_value: Random seed value
        seed_idx: Index in SEEDS list
        ... (other parameters same as Phase 3)
    
    Returns:
        Updated results dictionary
    """
    
    print(f"\n{'='*70}")
    print(f"Evaluating Seed {seed_value} ({seed_idx}/{len(SEEDS)})")
    print(f"{'='*70}")
    
    # Load existing result JSON
    result_file = os.path.join(METRICS_DIR, "experiments", f"seed_{seed_value}_results.json")
    if not os.path.exists(result_file):
        print(f"  ❌ Result file not found: {result_file}")
        return None
    
    with open(result_file, 'r') as f:
        results = json.load(f)
    
    # Check if per-class metrics already exist
    if 'per_class_metrics' in results.get('validation_metrics', {}) and \
       'per_class_metrics' in results.get('test_metrics', {}):
        print(f"  ⏭️  Per-class metrics already exist, skipping...")
        return results
    
    # Get data split seed from results
    data_split_seed = results['configuration']['data_split_seed']
    best_epoch = results['training_info']['best_epoch']
    
    print(f"  Data split seed: {data_split_seed}")
    print(f"  Best epoch: {best_epoch}")
    
    # Recreate data splits using the same seed
    print(f"  Recreating data splits with random_state={data_split_seed}...")
    train_df, temp_df = train_test_split(
        df_full,
        test_size=(VAL_RATIO + TEST_RATIO),
        stratify=df_full['style'],
        random_state=data_split_seed
    )
    
    val_df, test_df = train_test_split(
        temp_df,
        test_size=TEST_RATIO / (VAL_RATIO + TEST_RATIO),
        stratify=temp_df['style'],
        random_state=data_split_seed
    )
    
    print(f"  Split sizes: Train={len(train_df)}, Val={len(val_df)}, Test={len(test_df)}")
    
    # Image transforms
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    # Create datasets
    val_dataset = FashionMultiModalDataset(val_df, captions_dict, style_to_idx, transform, base_dir=base_dir)
    test_dataset = FashionMultiModalDataset(test_df, captions_dict, style_to_idx, transform, base_dir=base_dir)
    
    # Create data loaders
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    
    # Compute class weights (for criterion, but won't affect evaluation)
    class_weights = compute_class_weight(
        'balanced',
        classes=np.array(list(style_to_idx.values())),
        y=train_df['style'].map(style_to_idx).values
    )
    class_weights = torch.FloatTensor(class_weights).to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    
    # Initialize model (with fixed seed)
    torch.manual_seed(MODEL_INIT_SEED)
    np.random.seed(MODEL_INIT_SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(MODEL_INIT_SEED)
    
    model = MultiModalFashionClassifier(
        clip_model=clip_model,
        fashionbert_model=fashionbert_model,
        num_classes=num_classes,
        dropout=DROPOUT
    ).to(device)
    
    # Load saved model checkpoint
    model_path = os.path.join(ARTIFACTS_DIR, "models", f"seed_{seed_value}_best_model.pth")
    if not os.path.exists(model_path):
        print(f"  ❌ Model checkpoint not found: {model_path}")
        return None
    
    print(f"  Loading model from: {model_path}")
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    
    # Evaluate on validation set (at best epoch)
    print(f"  Evaluating on validation set...")
    (val_loss, val_acc, val_preds, val_labels,
     val_macro_f1, val_macro_precision, val_macro_recall,
     val_per_class_f1, val_per_class_precision, val_per_class_recall) = evaluate_model(
        model, val_loader, criterion, device, all_styles
    )
    
    # Evaluate on test set
    print(f"  Evaluating on test set...")
    (test_loss, test_acc, test_preds, test_labels,
     test_macro_f1, test_macro_precision, test_macro_recall,
     test_per_class_f1, test_per_class_precision, test_per_class_recall) = evaluate_model(
        model, test_loader, criterion, device, all_styles
    )
    
    # Update results with per-class metrics
    if 'validation_metrics' not in results:
        results['validation_metrics'] = {}
    if 'test_metrics' not in results:
        results['test_metrics'] = {}
    
    # Add per-class metrics to validation
    results['validation_metrics']['per_class_metrics'] = {
        'f1': val_per_class_f1,
        'precision': val_per_class_precision,
        'recall': val_per_class_recall
    }
    
    # Also update overall metrics (in case they differ slightly due to re-evaluation)
    results['validation_metrics']['best_val_macro_f1'] = float(val_macro_f1)
    results['validation_metrics']['best_val_macro_precision'] = float(val_macro_precision)
    results['validation_metrics']['best_val_macro_recall'] = float(val_macro_recall)
    results['validation_metrics']['best_val_accuracy'] = float(val_acc)
    
    # Add per-class metrics to test
    results['test_metrics']['per_class_metrics'] = {
        'f1': test_per_class_f1,
        'precision': test_per_class_precision,
        'recall': test_per_class_recall
    }
    
    # Update overall test metrics
    results['test_metrics']['test_macro_f1'] = float(test_macro_f1)
    results['test_metrics']['test_macro_precision'] = float(test_macro_precision)
    results['test_metrics']['test_macro_recall'] = float(test_macro_recall)
    results['test_metrics']['test_accuracy'] = float(test_acc)
    
    # Add timestamp for when per-class metrics were computed
    results['per_class_metrics_timestamp'] = datetime.now().isoformat()
    
    # Save updated results
    with open(result_file, 'w') as f:
        json.dump(results, f, indent=2)
    
    print(f"  ✅ Per-class metrics computed and saved!")
    print(f"     Val Macro F1: {val_macro_f1:.4f}, Test Macro F1: {test_macro_f1:.4f}")
    
    return results

print("✅ Evaluation function defined!")


✅ Evaluation function defined!


## 8. Process All Seeds


In [8]:
# Process all seeds
updated_results = []
skipped_seeds = []
error_seeds = []

print(f"\n{'='*70}")
print(f"STARTING PER-CLASS METRICS EVALUATION")
print(f"Total seeds: {len(SEEDS)}")
print(f"Estimated time: ~{len(SEEDS) * 2:.1f} minutes")
print(f"{'='*70}")

for seed_idx, seed_value in enumerate(SEEDS, 1):
    try:
        result = evaluate_seed_with_per_class_metrics(
            seed_value=seed_value,
            seed_idx=seed_idx,
            df_full=df_full,
            captions_dict=captions_dict,
            style_to_idx=style_to_idx,
            clip_model=clip_model,
            fashionbert_model=fashionbert_model,
            fashionbert_tokenizer=fashionbert_tokenizer,
            num_classes=num_classes,
            device=device,
            all_styles=all_styles,
            base_dir=BASE_DIR
        )
        
        if result is None:
            error_seeds.append(seed_value)
        elif 'per_class_metrics_timestamp' not in result:
            skipped_seeds.append(seed_value)
        else:
            updated_results.append(result)
        
        # Progress update
        remaining = len(SEEDS) - seed_idx
        print(f"\n✅ Progress: {seed_idx}/{len(SEEDS)} processed, {remaining} remaining")
        
    except Exception as e:
        print(f"\n❌ Error processing seed {seed_value}: {e}")
        error_seeds.append((seed_value, str(e)))
        import traceback
        traceback.print_exc()
        continue

print(f"\n{'='*70}")
print(f"EVALUATION COMPLETED!")
print(f"  Updated: {len(updated_results)}/{len(SEEDS)}")
print(f"  Skipped: {len(skipped_seeds)} (already had per-class metrics)")
print(f"  Errors: {len(error_seeds)}")
if error_seeds:
    print(f"  Error seeds: {error_seeds}")
print(f"{'='*70}")



STARTING PER-CLASS METRICS EVALUATION
Total seeds: 30
Estimated time: ~60.0 minutes

Evaluating Seed 13 (1/30)
  Data split seed: 13
  Best epoch: 12
  Recreating data splits with random_state=13...
  Split sizes: Train=9261, Val=1984, Test=1985
Dataset initialized with 1984 valid samples (out of 1984)
Dataset initialized with 1985 valid samples (out of 1985)
  Loading model from: phase3_robustness_results/models/seed_13_best_model.pth
  Evaluating on validation set...


  Evaluating on test set...


  ✅ Per-class metrics computed and saved!
     Val Macro F1: 0.8421, Test Macro F1: 0.8339

✅ Progress: 1/30 processed, 29 remaining

Evaluating Seed 14 (2/30)
  Data split seed: 14
  Best epoch: 9
  Recreating data splits with random_state=14...
  Split sizes: Train=9261, Val=1984, Test=1985
Dataset initialized with 1984 valid samples (out of 1984)
Dataset initialized with 1985 valid samples (out of 1985)
  Loading model from: phase3_robustness_results/models/seed_14_best_model.pth
  Evaluating on validation set...


  Evaluating on test set...


  ✅ Per-class metrics computed and saved!
     Val Macro F1: 0.8430, Test Macro F1: 0.8299

✅ Progress: 2/30 processed, 28 remaining

Evaluating Seed 16 (3/30)
  Data split seed: 16
  Best epoch: 16
  Recreating data splits with random_state=16...
  Split sizes: Train=9261, Val=1984, Test=1985
Dataset initialized with 1984 valid samples (out of 1984)
Dataset initialized with 1985 valid samples (out of 1985)
  Loading model from: phase3_robustness_results/models/seed_16_best_model.pth
  Evaluating on validation set...


...

  ✅ Per-class metrics computed and saved!
     Val Macro F1: 0.8370, Test Macro F1: 0.8216

✅ Progress: 28/30 processed, 2 remaining

Evaluating Seed 380 (29/30)
  Data split seed: 380
  Best epoch: 13
  Recreating data splits with random_state=380...
  Split sizes: Train=9261, Val=1984, Test=1985
Dataset initialized with 1984 valid samples (out of 1984)
Dataset initialized with 1985 valid samples (out of 1985)
  Loading model from: phase3_robustness_results/models/seed_380_best_model.pth
  Evaluating on validation set...


  Evaluating on test set...


  ✅ Per-class metrics computed and saved!
     Val Macro F1: 0.8335, Test Macro F1: 0.8562

✅ Progress: 29/30 processed, 1 remaining

Evaluating Seed 457 (30/30)
  Data split seed: 457
  Best epoch: 6
  Recreating data splits with random_state=457...
  Split sizes: Train=9261, Val=1984, Test=1985
Dataset initialized with 1984 valid samples (out of 1984)
Dataset initialized with 1985 valid samples (out of 1985)
  Loading model from: phase3_robustness_results/models/seed_457_best_model.pth
  Evaluating on validation set...


  Evaluating on test set...


  ✅ Per-class metrics computed and saved!
     Val Macro F1: 0.8366, Test Macro F1: 0.8283

✅ Progress: 30/30 processed, 0 remaining

EVALUATION COMPLETED!
  Updated: 30/30
  Skipped: 0 (already had per-class metrics)
  Errors: 0


## 9. Load All Results and Compute Statistics


In [9]:
# Load all results (with per-class metrics)
print("Loading all results with per-class metrics...")
all_results = []
for seed_value in SEEDS:
    result_file = os.path.join(METRICS_DIR, "experiments", f"seed_{seed_value}_results.json")
    if os.path.exists(result_file):
        with open(result_file, 'r') as f:
            result = json.load(f)
            # Check if per-class metrics exist
            if 'per_class_metrics' in result.get('validation_metrics', {}) and \
               'per_class_metrics' in result.get('test_metrics', {}):
                all_results.append(result)
            else:
                print(f"  ⚠️  Seed {seed_value} missing per-class metrics")

print(f"✅ Loaded {len(all_results)} results with per-class metrics")

if len(all_results) == 0:
    print("\n❌ No results with per-class metrics found!")
    print("   Please run the evaluation cells above first.")


Loading all results with per-class metrics...
✅ Loaded 30 results with per-class metrics


## 10. Statistical Analysis: Overall Metrics


In [10]:
def calculate_stats(values, name):
    """Calculate comprehensive statistics for a metric"""
    mean_val = np.mean(values)
    std_val = np.std(values)
    min_val = np.min(values)
    max_val = np.max(values)
    median_val = np.median(values)
    q25 = np.percentile(values, 25)
    q75 = np.percentile(values, 75)
    cv = (std_val / mean_val * 100) if mean_val != 0 else 0  # Coefficient of Variation
    
    # 95% Confidence Interval
    if len(values) > 1:
        ci = stats.t.interval(0.95, len(values)-1, loc=mean_val, scale=stats.sem(values))
        ci_lower, ci_upper = ci
    else:
        ci_lower, ci_upper = mean_val, mean_val
    
    return {
        "metric": name,
        "mean": float(mean_val),
        "std": float(std_val),
        "min": float(min_val),
        "max": float(max_val),
        "median": float(median_val),
        "q25": float(q25),
        "q75": float(q75),
        "cv_percent": float(cv),
        "ci_95_lower": float(ci_lower),
        "ci_95_upper": float(ci_upper),
        "n": len(values)
    }

if len(all_results) > 0:
    # Extract overall metrics
    test_f1s = [r['test_metrics']['test_macro_f1'] for r in all_results]
    test_accs = [r['test_metrics']['test_accuracy'] for r in all_results]
    test_precisions = [r['test_metrics'].get('test_macro_precision', 0) for r in all_results]
    test_recalls = [r['test_metrics'].get('test_macro_recall', 0) for r in all_results]
    
    best_val_f1s = [r['validation_metrics']['best_val_macro_f1'] for r in all_results]
    best_val_accs = [r['validation_metrics']['best_val_accuracy'] for r in all_results]
    
    # Create statistics table for overall metrics
    overall_stats = [
        calculate_stats(test_f1s, "Test Macro F1"),
        calculate_stats(test_accs, "Test Accuracy"),
        calculate_stats(test_precisions, "Test Macro Precision"),
        calculate_stats(test_recalls, "Test Macro Recall"),
        calculate_stats(best_val_f1s, "Best Val Macro F1"),
        calculate_stats(best_val_accs, "Best Val Accuracy")
    ]
    
    df_overall_stats = pd.DataFrame(overall_stats)
    
    print("\n" + "="*80)
    print("OVERALL METRICS - Statistical Summary (30 runs)")
    print("="*80)
    print(df_overall_stats.to_string(index=False))
    
    # Save overall statistics
    overall_stats_path = os.path.join(METRICS_DIR, "overall_metrics_statistics.json")
    with open(overall_stats_path, 'w') as f:
        json.dump({"overall_metrics": overall_stats}, f, indent=2)
    
    df_overall_stats.to_csv(os.path.join(METRICS_DIR, "overall_metrics_summary.csv"), index=False)
    
    print(f"\n✅ Overall statistics saved to:")
    print(f"   - {overall_stats_path}")
    print(f"   - {os.path.join(METRICS_DIR, 'overall_metrics_summary.csv')}")
else:
    print("⚠️  No results available for statistical analysis")



OVERALL METRICS - Statistical Summary (30 runs)
              metric      mean      std       min       max    median       q25       q75  cv_percent  ci_95_lower  ci_95_upper  n
       Test Macro F1  0.831084 0.009444  0.806897  0.856231  0.830406  0.825812  0.837092    1.136395     0.827497     0.834671 30
       Test Accuracy 83.118388 0.925120 80.705290 85.591940 83.047859 82.518892 83.727960    1.113015    82.767037    83.469739 30
Test Macro Precision  0.832373 0.009255  0.808782  0.856024  0.831956  0.826588  0.837407    1.111896     0.828858     0.835888 30
   Test Macro Recall  0.833802 0.009483  0.809633  0.859154  0.833790  0.828003  0.838861    1.137294     0.830200     0.837403 30
   Best Val Macro F1  0.837256 0.007406  0.819662  0.853017  0.837426  0.833610  0.841608    0.884566     0.834443     0.840069 30
   Best Val Accuracy 83.726478 0.726895 82.006048 85.332661 83.719758 83.329133 84.248992    0.868178    83.450411    84.002546 30

✅ Overall statistics saved to:
  

## 11. Statistical Analysis: Per-Class Metrics


In [11]:
if len(all_results) > 0:
    # Extract per-class metrics for each style
    per_class_stats = {}
    
    for style in all_styles:
        # Extract F1, Precision, Recall for this style across all runs
        test_f1s_style = []
        test_precisions_style = []
        test_recalls_style = []
        
        val_f1s_style = []
        val_precisions_style = []
        val_recalls_style = []
        
        for result in all_results:
            # Test metrics
            test_pc = result['test_metrics'].get('per_class_metrics', {})
            if test_pc and 'f1' in test_pc and style in test_pc['f1']:
                test_f1s_style.append(test_pc['f1'][style])
                test_precisions_style.append(test_pc['precision'][style])
                test_recalls_style.append(test_pc['recall'][style])
            
            # Validation metrics
            val_pc = result['validation_metrics'].get('per_class_metrics', {})
            if val_pc and 'f1' in val_pc and style in val_pc['f1']:
                val_f1s_style.append(val_pc['f1'][style])
                val_precisions_style.append(val_pc['precision'][style])
                val_recalls_style.append(val_pc['recall'][style])
        
        # Calculate statistics for this style
        per_class_stats[style] = {
            'test': {
                'f1': calculate_stats(test_f1s_style, f"{style} - Test F1") if test_f1s_style else None,
                'precision': calculate_stats(test_precisions_style, f"{style} - Test Precision") if test_precisions_style else None,
                'recall': calculate_stats(test_recalls_style, f"{style} - Test Recall") if test_recalls_style else None
            },
            'validation': {
                'f1': calculate_stats(val_f1s_style, f"{style} - Val F1") if val_f1s_style else None,
                'precision': calculate_stats(val_precisions_style, f"{style} - Val Precision") if val_precisions_style else None,
                'recall': calculate_stats(val_recalls_style, f"{style} - Val Recall") if val_recalls_style else None
            }
        }
    
    # Create per-class summary table (Test F1-Score)
    per_class_data = []
    for style in all_styles:
        if per_class_stats[style]['test']['f1']:
            stats = per_class_stats[style]['test']['f1']
            per_class_data.append({
                'Style': style,
                'Mean': stats['mean'],
                'Std': stats['std'],
                'Min': stats['min'],
                'Max': stats['max'],
                'CI_95_Lower': stats['ci_95_lower'],
                'CI_95_Upper': stats['ci_95_upper'],
                'CV_%': stats['cv_percent']
            })
    
    df_per_class = pd.DataFrame(per_class_data)
    
    print("\n" + "="*80)
    print("PER-CLASS METRICS - Test F1-Score (30 runs)")
    print("="*80)
    print(df_per_class.to_string(index=False))
    
    # Save per-class statistics
    per_class_stats_path = os.path.join(METRICS_DIR, "per_class_metrics_statistics.json")
    with open(per_class_stats_path, 'w') as f:
        json.dump(per_class_stats, f, indent=2)
    
    df_per_class.to_csv(os.path.join(METRICS_DIR, "per_class_metrics_summary.csv"), index=False)
    
    print(f"\n✅ Per-class statistics saved to:")
    print(f"   - {per_class_stats_path}")
    print(f"   - {os.path.join(METRICS_DIR, 'per_class_metrics_summary.csv')}")
else:
    print("⚠️  No results available for per-class analysis")



PER-CLASS METRICS - Test F1-Score (30 runs)
         Style     Mean      Std      Min      Max  CI_95_Lower  CI_95_Upper     CV_%
  conservative 0.791048 0.021962 0.751880 0.837209     0.782707     0.799389 2.776333
        dressy 0.956387 0.011897 0.931408 0.981550     0.951868     0.960905 1.243959
        ethnic 0.842944 0.021635 0.802867 0.877470     0.834728     0.851161 2.566559
         fairy 0.931537 0.012622 0.903226 0.951724     0.926743     0.936331 1.355001
      feminine 0.762263 0.027556 0.712000 0.813278     0.751797     0.772728 3.614990
           gal 0.863894 0.014722 0.836013 0.897059     0.858303     0.869485 1.704109
       girlish 0.694649 0.025228 0.640569 0.750000     0.685068     0.704230 3.631702
kireime-casual 0.722027 0.022147 0.671141 0.767677     0.713616     0.730439 3.067335
        lolita 0.930508 0.018167 0.872274 0.968354     0.923608     0.937407 1.952388
          mode 0.806594 0.024870 0.748299 0.861736     0.797149     0.816039 3.083287
       na

## 12. Generate Comprehensive Report Tables


In [12]:
if len(all_results) > 0:
    # Create formatted summary table for overall metrics
    print("\n" + "="*80)
    print("COMPREHENSIVE REPORT: Overall Performance (30 runs)")
    print("="*80)
    
    # Format: Metric | Mean ± Std | 95% CI | Min | Max | CV%
    report_data = []
    for stats in overall_stats:
        report_data.append({
            'Metric': stats['metric'],
            'Mean ± Std': f"{stats['mean']:.4f} ± {stats['std']:.4f}",
            '95% CI': f"[{stats['ci_95_lower']:.4f}, {stats['ci_95_upper']:.4f}]",
            'Min': f"{stats['min']:.4f}",
            'Max': f"{stats['max']:.4f}",
            'CV %': f"{stats['cv_percent']:.2f}%"
        })
    
    df_report = pd.DataFrame(report_data)
    print(df_report.to_string(index=False))
    
    # Create formatted per-class table
    print("\n" + "="*80)
    print("COMPREHENSIVE REPORT: Per-Class Performance (30 runs)")
    print("="*80)
    
    # Format: Style | F1 (Mean ± Std) | Precision (Mean ± Std) | Recall (Mean ± Std)
    per_class_report_data = []
    for style in all_styles:
        if per_class_stats[style]['test']['f1']:
            f1_stats = per_class_stats[style]['test']['f1']
            prec_stats = per_class_stats[style]['test']['precision']
            rec_stats = per_class_stats[style]['test']['recall']
            
            per_class_report_data.append({
                'Style': style,
                'F1-Score': f"{f1_stats['mean']:.4f} ± {f1_stats['std']:.4f}",
                'Precision': f"{prec_stats['mean']:.4f} ± {prec_stats['std']:.4f}",
                'Recall': f"{rec_stats['mean']:.4f} ± {rec_stats['std']:.4f}",
                '95% CI (F1)': f"[{f1_stats['ci_95_lower']:.4f}, {f1_stats['ci_95_upper']:.4f}]"
            })
    
    df_per_class_report = pd.DataFrame(per_class_report_data)
    print(df_per_class_report.to_string(index=False))
    
    # Save comprehensive reports
    df_report.to_csv(os.path.join(METRICS_DIR, "comprehensive_report_overall.csv"), index=False)
    df_per_class_report.to_csv(os.path.join(METRICS_DIR, "comprehensive_report_per_class.csv"), index=False)
    
    print(f"\n✅ Comprehensive reports saved to:")
    print(f"   - {os.path.join(METRICS_DIR, 'comprehensive_report_overall.csv')}")
    print(f"   - {os.path.join(METRICS_DIR, 'comprehensive_report_per_class.csv')}")
else:
    print("⚠️  No results available for report generation")



COMPREHENSIVE REPORT: Overall Performance (30 runs)
              Metric       Mean ± Std             95% CI     Min     Max  CV %
       Test Macro F1  0.8311 ± 0.0094   [0.8275, 0.8347]  0.8069  0.8562 1.14%
       Test Accuracy 83.1184 ± 0.9251 [82.7670, 83.4697] 80.7053 85.5919 1.11%
Test Macro Precision  0.8324 ± 0.0093   [0.8289, 0.8359]  0.8088  0.8560 1.11%
   Test Macro Recall  0.8338 ± 0.0095   [0.8302, 0.8374]  0.8096  0.8592 1.14%
   Best Val Macro F1  0.8373 ± 0.0074   [0.8344, 0.8401]  0.8197  0.8530 0.88%
   Best Val Accuracy 83.7265 ± 0.7269 [83.4504, 84.0025] 82.0060 85.3327 0.87%

COMPREHENSIVE REPORT: Per-Class Performance (30 runs)
         Style        F1-Score       Precision          Recall      95% CI (F1)
  conservative 0.7910 ± 0.0220 0.7742 ± 0.0302 0.8115 ± 0.0482 [0.7827, 0.7994]
        dressy 0.9564 ± 0.0119 0.9586 ± 0.0204 0.9546 ± 0.0153 [0.9519, 0.9609]
        ethnic 0.8429 ± 0.0216 0.8283 ± 0.0326 0.8597 ± 0.0336 [0.8347, 0.8512]
         fairy 0.93

## 13. Visualizations


In [13]:
if len(all_results) > 0 and len(per_class_stats) > 0:
    # Create output directory for visualizations
    viz_dir = os.path.join(ARTIFACTS_DIR, "per_class_visualizations")
    os.makedirs(viz_dir, exist_ok=True)
    
    # Plot 1: Per-Class F1-Score with Error Bars
    styles_list = []
    f1_means = []
    f1_stds = []
    
    for style in all_styles:
        if per_class_stats[style]['test']['f1']:
            styles_list.append(style)
            f1_means.append(per_class_stats[style]['test']['f1']['mean'])
            f1_stds.append(per_class_stats[style]['test']['f1']['std'])
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # Plot 1: Per-Class F1-Score Bar Chart with Error Bars
    x_pos = np.arange(len(styles_list))
    axes[0, 0].bar(x_pos, f1_means, yerr=f1_stds, capsize=5, alpha=0.7, color='skyblue', edgecolor='black')
    axes[0, 0].set_xlabel('Style')
    axes[0, 0].set_ylabel('F1-Score (Mean ± Std)')
    axes[0, 0].set_title('Per-Class F1-Score (Test Set, 30 runs)', fontsize=12, fontweight='bold')
    axes[0, 0].set_xticks(x_pos)
    axes[0, 0].set_xticklabels(styles_list, rotation=45, ha='right')
    axes[0, 0].grid(True, alpha=0.3, axis='y')
    axes[0, 0].axhline(y=np.mean(f1_means), color='red', linestyle='--', alpha=0.7, label=f'Mean: {np.mean(f1_means):.4f}')
    axes[0, 0].legend()
    
    # Plot 2: Per-Class Precision vs Recall
    prec_means = []
    rec_means = []
    for style in styles_list:
        prec_means.append(per_class_stats[style]['test']['precision']['mean'])
        rec_means.append(per_class_stats[style]['test']['recall']['mean'])
    
    axes[0, 1].scatter(prec_means, rec_means, s=100, alpha=0.6, edgecolors='black', linewidth=1)
    for i, style in enumerate(styles_list):
        axes[0, 1].annotate(style, (prec_means[i], rec_means[i]), fontsize=8, ha='center')
    axes[0, 1].set_xlabel('Precision (Mean)')
    axes[0, 1].set_ylabel('Recall (Mean)')
    axes[0, 1].set_title('Precision vs Recall (Test Set, 30 runs)', fontsize=12, fontweight='bold')
    axes[0, 1].grid(True, alpha=0.3)
    axes[0, 1].plot([0, 1], [0, 1], 'k--', alpha=0.3)  # Diagonal line
    
    # Plot 3: Coefficient of Variation (CV) for each class
    cv_values = []
    for style in styles_list:
        cv_values.append(per_class_stats[style]['test']['f1']['cv_percent'])
    
    axes[1, 0].bar(x_pos, cv_values, alpha=0.7, color='lightcoral', edgecolor='black')
    axes[1, 0].set_xlabel('Style')
    axes[1, 0].set_ylabel('Coefficient of Variation (%)')
    axes[1, 0].set_title('Per-Class F1 CV% (Lower = More Stable)', fontsize=12, fontweight='bold')
    axes[1, 0].set_xticks(x_pos)
    axes[1, 0].set_xticklabels(styles_list, rotation=45, ha='right')
    axes[1, 0].grid(True, alpha=0.3, axis='y')
    axes[1, 0].axhline(y=np.mean(cv_values), color='red', linestyle='--', alpha=0.7, label=f'Mean CV: {np.mean(cv_values):.2f}%')
    axes[1, 0].legend()
    
    # Plot 4: Heatmap of Per-Class Metrics
    metrics_matrix = []
    for style in styles_list:
        metrics_matrix.append([
            per_class_stats[style]['test']['f1']['mean'],
            per_class_stats[style]['test']['precision']['mean'],
            per_class_stats[style]['test']['recall']['mean']
        ])
    
    metrics_matrix = np.array(metrics_matrix)
    sns.heatmap(metrics_matrix, 
                xticklabels=['F1', 'Precision', 'Recall'],
                yticklabels=styles_list,
                annot=True, fmt='.3f', cmap='YlOrRd',
                cbar_kws={'label': 'Score'}, ax=axes[1, 1])
    axes[1, 1].set_title('Per-Class Metrics Heatmap (Test Set, 30 runs)', fontsize=12, fontweight='bold')
    axes[1, 1].set_ylabel('Style')
    
    plt.suptitle('Per-Class Performance Analysis (30 Robustness Experiments)', fontsize=16, fontweight='bold')
    plt.tight_layout()
    
    # Save plot
    plot_path = os.path.join(viz_dir, "per_class_metrics_analysis.png")
    plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"✅ Per-class visualizations saved to: {plot_path}")
    
    # Additional plot: Box plot of per-class F1 distribution
    fig, ax = plt.subplots(figsize=(14, 8))
    
    # Extract F1 values for each class across all runs
    f1_data_by_class = []
    for style in styles_list:
        f1_values = []
        for result in all_results:
            test_pc = result['test_metrics'].get('per_class_metrics', {})
            if test_pc and 'f1' in test_pc and style in test_pc['f1']:
                f1_values.append(test_pc['f1'][style])
        f1_data_by_class.append(f1_values)
    
    bp = ax.boxplot(f1_data_by_class, labels=styles_list, patch_artist=True,
                    boxprops=dict(facecolor='lightblue', alpha=0.7),
                    medianprops=dict(color='red', linewidth=2))
    ax.set_xlabel('Style', fontsize=12)
    ax.set_ylabel('F1-Score', fontsize=12)
    ax.set_title('Per-Class F1-Score Distribution (30 runs)', fontsize=14, fontweight='bold')
    ax.tick_params(axis='x', rotation=45)
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    boxplot_path = os.path.join(viz_dir, "per_class_f1_distribution.png")
    plt.savefig(boxplot_path, dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"✅ F1 distribution box plot saved to: {boxplot_path}")
else:
    print("⚠️  No results available for visualization")


✅ Per-class visualizations saved to: phase3_robustness_results/per_class_visualizations/per_class_metrics_analysis.png
✅ F1 distribution box plot saved to: phase3_robustness_results/per_class_visualizations/per_class_f1_distribution.png


## 14. Final Summary


In [14]:
if len(all_results) > 0:
    print("\n" + "="*80)
    print("FINAL SUMMARY - Phase 3 Per-Class Metrics Analysis")
    print("="*80)
    
    # Overall performance
    test_f1_mean = np.mean([r['test_metrics']['test_macro_f1'] for r in all_results])
    test_f1_std = np.std([r['test_metrics']['test_macro_f1'] for r in all_results])
    test_acc_mean = np.mean([r['test_metrics']['test_accuracy'] for r in all_results])
    test_acc_std = np.std([r['test_metrics']['test_accuracy'] for r in all_results])
    
    print(f"\n📊 PRIMARY METRIC (Test Macro F1):")
    print(f"   {test_f1_mean:.4f} ± {test_f1_std:.4f}")
    
    print(f"\n📈 Overall Test Performance:")
    print(f"   Accuracy: {test_acc_mean:.2f}% ± {test_acc_std:.2f}%")
    
    # Per-class summary
    if len(per_class_stats) > 0:
        print(f"\n📋 Per-Class Performance Summary:")
        print(f"   Best performing class: {max(styles_list, key=lambda s: per_class_stats[s]['test']['f1']['mean'])}")
        print(f"   Worst performing class: {min(styles_list, key=lambda s: per_class_stats[s]['test']['f1']['mean'])}")
        print(f"   Most stable class (lowest CV): {min(styles_list, key=lambda s: per_class_stats[s]['test']['f1']['cv_percent'])}")
        print(f"   Least stable class (highest CV): {max(styles_list, key=lambda s: per_class_stats[s]['test']['f1']['cv_percent'])}")
    
    print(f"\n✅ Outputs under: {EXPERIMENT_ROOT}")
    print(f"   - Overall metrics: overall_metrics_summary.csv")
    print(f"   - Per-class metrics: per_class_metrics_summary.csv")
    print(f"   - Comprehensive reports: comprehensive_report_*.csv")
    print(f"   - Visualizations: per_class_visualizations/")
    
    print("\n" + "="*80)
else:
    print("⚠️  No results available for final summary")



FINAL SUMMARY - Phase 3 Per-Class Metrics Analysis

📊 PRIMARY METRIC (Test Macro F1):
   0.8311 ± 0.0094

📈 Overall Test Performance:
   Accuracy: 83.12% ± 0.93%

📋 Per-Class Performance Summary:
   Best performing class: dressy
   Worst performing class: girlish
   Most stable class (lowest CV): dressy
   Least stable class (highest CV): girlish

✅ All results saved to: phase3_robustness_results
   - Overall metrics: overall_metrics_summary.csv
   - Per-class metrics: per_class_metrics_summary.csv
   - Comprehensive reports: comprehensive_report_*.csv
   - Visualizations: per_class_visualizations/

